# Intel Welding — open one real welding sample

A beginner-friendly, read-only tour: [overview](#overview) → [one sample](#sample) → [frames](#frames) → [sensors](#sensors) → [audio](#audio) → [shapes and code](#trace).

A **session** groups related welds (`DIRECTORY` in the manifest). A **sample** is one sample folder (`SUBDIRS`), containing associated video, audio and sensor files where available. A **modality** is one observation type; a video **frame** is one image; a sensor **channel** is one measured quantity. A **label** is an assigned category. An **array** is a block of numbers, and its **shape** counts observations and components.

`CATEGORY` records the **intended weld condition**, not an expert-verified physical defect. The research contract’s proposed binary target is Good versus non-Good. This notebook does not fit that target, diagnose a defect, or claim that the example visibly proves its label.

We use the existing foundation audit `20260927T114705163655Z`. Its alignment status is conditional/unvalidated and its proposed final holdout was invalidated. We select only from the recorded development-session list, keep `allow_final_test` false, and perform no final-test signal inspection or modelling.

In [ ]:
DOMAIN = 'intel_welding'
SAMPLE_SUBDIR = None  # Optional exact manifest SUBDIRS string from development only
AUDIO_SECONDS = 5  # bounded first excerpt; not aligned to the video or sensor clock
from pathlib import Path
import os, sys, json, importlib.util
from datetime import datetime, timezone
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'REPO_MAP.md').exists())
sys.dont_write_bytecode = True
sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT/'tutorials/EDA'))
from reassemble_support import identity, reference
if 'EDA_OUTPUT' in os.environ:
    OUT = Path(os.environ['EDA_OUTPUT'])
else:
    OUT = ROOT/'runs'/DOMAIN/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')+'-dataset-eda')
    OUT.mkdir(parents=True, exist_ok=False)
    (OUT/'.gitignore').write_text('*\n')
for name in ['figures','media','tables','reports','manifests','config','runtime']:
    (OUT/name).mkdir(exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(OUT/'runtime/matplotlib')
import numpy as np, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image, Audio
plt.rcParams.update({'font.size':11, 'axes.spines.top':False, 'axes.spines.right':False})
inputs = {}; warnings = []; shapes = []
def track(path):
    inputs[os.path.relpath(path,ROOT)] = identity(path,ROOT,Path(path).stat().st_size < 20*1024**2)
def table(frame): display(frame.style.hide(axis='index'))
def figure(fig,name):
    path=OUT/'figures'/name
    fig.savefig(path,dpi=120,bbox_inches='tight');plt.close(fig)
    display(Image(filename=str(path)))
def shape(stage,array,meaning):
    shapes.append(dict(stage=stage,shape=str(array.shape),dtype=str(array.dtype),dimensions=meaning,evidence='Measured from loaded array'))
def module(name,path):
    spec=importlib.util.spec_from_file_location(name,path);m=importlib.util.module_from_spec(spec)
    sys.modules[name]=m;spec.loader.exec_module(m);track(path);return m

<a id="overview"></a>
## 1. What exists, and what was inventoried?

The raw root is resolved from `configs/intel_welding/foundations.json`, not from PHM configuration. Overview counts come from the saved inventory; only the selected sample’s files are inspected now.

In [ ]:
cfg_path=ROOT/'configs/intel_welding/foundations.json';cfg=json.loads(cfg_path.read_text());track(cfg_path)
RAW=Path(os.environ.get('INTEL_WELDING_DATA_ROOT',str(ROOT/cfg['data_root'])))
AUDIT=ROOT/'artifacts/intel_welding/audits/foundation_runs/20260927T114705163655Z'
inv_path=AUDIT/'dataset_inventory.parquet';inv=pd.read_parquet(inv_path);track(inv_path)
split_path=ROOT/cfg['existing_split'];split=json.loads(split_path.read_text());track(split_path)
assert cfg['allow_final_test'] is False
assert not set(split['development_sessions'])&set(split['final_test_sessions'])
manifest_path=RAW/'manifest.csv'
manifest=pd.read_csv(manifest_path);track(manifest_path)
assert not inv.SUBDIRS.duplicated().any()
assert set(inv.SUBDIRS)==set(manifest.SUBDIRS)
table(pd.DataFrame([['Configured raw root',cfg['data_root'],RAW.exists()],['Saved foundation inventory',str(inv_path.relative_to(ROOT)),inv_path.exists()]],columns=['role','path','exists now']))
table(pd.DataFrame([['Manifest sample rows',len(manifest)],['Saved inventory rows',len(inv)],['Saved inventory sessions',inv.session_id.nunique()],['All three process modalities present (saved)',int(inv.complete_present.sum())],['All three process modalities readable (saved)',int(inv.complete_readable.sum())]],columns=['population','count']))
counts=inv.groupby('CATEGORY').agg(samples=('SUBDIRS','size'),sessions=('session_id','nunique')).sort_index()
table(counts.reset_index())
fig,ax=plt.subplots(figsize=(11,5));counts.samples.plot.barh(ax=ax,color='#367fa5')
ax.set(title='Saved foundation inventory: intended weld conditions',xlabel='Sample folders (count)',ylabel='Manifest CATEGORY')
figure(fig,'intended_conditions.png')

**What you are seeing:** recorded intended conditions, counted over the saved local inventory. Session counts remain separate because repeated welds within a session are not independent collection conditions.

**Why it matters:** label/session confounding and missing modalities affect what can be learned. `SPLIT` supplied with the dataset is not the protected session-based development/test assignment. No category is relabelled or merged here.

<a id="sample"></a>
## 2. Select one development sample

Rule: choose the lexicographically first composite `SUBDIRS` among saved complete/readable development samples with valid sensor timestamps and currently present modality files. This avoids selection by model prediction or visual appearance. Set `SAMPLE_SUBDIR` above to choose another eligible sample.

In [ ]:
eligible=inv[inv.session_id.isin(split['development_sessions']) & inv.complete_readable & inv.timestamp_valid.fillna(False)].copy()
eligible=eligible[eligible.apply(lambda r:all((RAW/r[k]).is_file() for k in ['video_path','audio_path','sensor_path']),axis=1)].sort_values('SUBDIRS')
assert len(eligible)>0,'No complete local development sample'
if SAMPLE_SUBDIR is None:row=eligible.iloc[0]
else:
    matches=eligible[eligible.SUBDIRS==SAMPLE_SUBDIR];assert len(matches)==1;row=matches.iloc[0]
assert row.session_id not in split['final_test_sessions']
original=manifest.loc[manifest.SUBDIRS==row.SUBDIRS].iloc[0]
assert original.CATEGORY==row.CATEGORY
sample_identity={'session_id':row.session_id,'sample_id':row.sample_id,'SUBDIRS':row.SUBDIRS,'CATEGORY':row.CATEGORY,'proposed_binary_target_non_good':int(row.CATEGORY!='Good')}
table(pd.DataFrame(sample_identity.items(),columns=['field','value']))
table(pd.DataFrame([original]))
folder=RAW/row.SUBDIRS
files=[]
for path in sorted(folder.iterdir()):
    files.append({'file':path.name,'kind':'directory' if path.is_dir() else path.suffix,'bytes':path.stat().st_size if path.is_file() else None})
table(pd.DataFrame(files))
paths={kind:RAW/row[key] for kind,key in [('video','video_path'),('audio','audio_path'),('sensor','sensor_path')]}
for path in paths.values():
    assert path.resolve().is_relative_to(RAW.resolve());track(path)

| Stream | What it contains | Use in this walkthrough |
|---|---|---|
| `.avi` | Video of the welding process | Four real frame previews |
| `.csv` | Timestamped process measurements | One sample’s sensor table and channels |
| `.flac` | Recorded audio | First five seconds, waveform and optional playback |
| Post-weld images, when present | Offline inspection photographs | Presence reported; not a deployable in-process input |
| Manifest fields | Intended condition and process settings | Context and identity, not measured sensor truth |

Files in the same sample folder are associated at **sample level**. Their starts, lengths and clocks can differ. No exact cross-modal synchronization is claimed.

<a id="frames"></a>
## 3. Real frames from this weld

The four frames are presentation previews, selected at evenly spaced recording-relative indices. Container frame/FPS time is nominal video time; it is not a verified shared timestamp with the CSV or audio.

In [ ]:
import cv2
cv2.setNumThreads(1)
cap=cv2.VideoCapture(str(paths['video']));assert cap.isOpened()
video_count=int(cap.get(cv2.CAP_PROP_FRAME_COUNT));fps=cap.get(cv2.CAP_PROP_FPS);assert fps>0
indices=np.linspace(0,video_count-1,4).round().astype(int);frames=[]
try:
    for index in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(index));ok,img=cap.read();assert ok
        frames.append(cv2.cvtColor(img,cv2.COLOR_BGR2RGB))
finally:cap.release()
stack=np.stack(frames);shape('Decoded video frame',stack[0],'height × width × RGB channels')
shape('Presentation frame stack',stack,'4 chosen frames × height × width × RGB channels')
fig,axes=plt.subplots(1,4,figsize=(15,4),layout='constrained')
for ax,index,img in zip(axes,indices,frames):
    ax.imshow(img);ax.axis('off');ax.set_title(f'frame {index}\nnominal {index/fps:.2f} s')
fig.suptitle(row.SUBDIRS+' — intended '+row.CATEGORY,fontsize=11)
figure(fig,'sample_frames.png')
print('Container metadata:',{'frames':video_count,'fps':fps,'nominal_seconds':video_count/fps})
print('Saved post-weld image count for this sample:',int(row.n_images))

**What you are seeing:** actual pixels from one identified welding video, with source frame indices.

**Why it matters:** the intended category may not be visibly diagnosable from these views. Four decoded frames demonstrate bounded readability; they do not certify every frame or establish a model-selected frame set.

<a id="sensors"></a>
## 4. Sensor rows from the same sample folder

We use the existing foundation `timing` function to check Date+Time. Channel names come directly from the CSV. Physical units are not inferred from similarly named manifest recipe settings. A missing value stays missing; no interpolation or resampling is performed.

In [ ]:
foundation=module('intel_foundation_eda',ROOT/'scripts/intel_welding/04_foundations.py')
assert paths['sensor'].stat().st_size<10*1024**2
sensor=pd.read_csv(paths['sensor']);sensor.columns=sensor.columns.str.strip()
timing=foundation.timing(sensor)
table(sensor.head(6));table(pd.DataFrame(timing.items(),columns=['timing check','value']))
assert timing['timestamp_valid'],'Do not draw a time series with invalid ordering'
t=pd.to_datetime(sensor.Date.astype(str).str.strip()+' '+sensor.Time.astype(str).str.strip(),format='%m/%d/%Y %H:%M:%S.%f',errors='raise')
seconds=(t-t.iloc[0]).dt.total_seconds()
channels=['Primary Weld Current','Secondary Weld Voltage']
values=sensor[channels].to_numpy();shape('Selected native sensor channels',values,'CSV observations × 2 named channels')
fig,axes=plt.subplots(2,1,figsize=(11,6),sharex=True,layout='constrained')
for ax,name in zip(axes,channels):
    ax.plot(seconds,sensor[name],'.',markersize=3);ax.set_ylabel(name+'\nunit unverified');ax.grid(alpha=.2)
axes[0].set_title('Actual native observations — '+row.sample_id)
axes[-1].set_xlabel('Seconds since first CSV timestamp (not the video origin)')
figure(fig,'sample_sensors.png')
table(pd.DataFrame({'channel':sensor.columns,'missing cells':sensor.isna().sum().to_numpy()}))

**What you are seeing:** native current/voltage observations for the selected sample, on the CSV’s own clock. Dots avoid drawing continuous curves across gaps.

**Why it matters:** rows occur at measured timestamp intervals, not assumed uniform video rate. The reported rate uses positive adjacent time differences. Missing `Remarks` entries, for example, are not automatically missing current/voltage. Identifiers, global timestamps and remarks are not proposed predictive channels.

<a id="audio"></a>
## 5. A short audio excerpt — same sample, separate clock

The first `AUDIO_SECONDS` seconds are decoded from this sample’s FLAC. This is an illustrative excerpt with a local audio clock; we do not align it with an arc frame or sensor event. Playback is optional and the waveform is the static fallback.

In [ ]:
import soundfile as sf
with sf.SoundFile(paths['audio']) as audio:
    rate=audio.samplerate;audio_total=len(audio);audio_channels=audio.channels
    excerpt=audio.read(min(len(audio),int(AUDIO_SECONDS*rate)),dtype='float32',always_2d=True)
shape('Decoded audio excerpt',excerpt,'audio samples × audio channels')
fig,ax=plt.subplots(figsize=(11,3.5))
stride=max(1,len(excerpt)//12000)
ax.plot(np.arange(len(excerpt))[::stride]/rate,excerpt[::stride,0],lw=.5)
ax.set(title=f'First {len(excerpt)/rate:.2f} seconds of audio, channel 0',xlabel='Seconds since decoded audio start',ylabel='Decoded amplitude (not calibrated pressure)')
figure(fig,'sample_audio.png')
display(Audio(excerpt.T,rate=rate,normalize=False))
table(pd.DataFrame([['Audio',audio_total/rate],['Video (container estimate)',video_count/fps],['Sensor CSV',timing['sensor_seconds']]],columns=['clock/stream','duration seconds']))
warnings.append('Cross-modal alignment is not validated; duration differences are not synchronization offsets.')
warnings.append('Audio/browser playback was not manually tested; decoded samples and a static waveform are available.')

**What you are seeing:** actual decoded audio amplitude and three stream durations.

**Why it matters:** matching file identity does not establish matching zero times or exact synchronization. A duration difference is not a measured lag. No lag correction, trimming-to-align, or temporal fusion is performed.

<a id="trace"></a>
## 6. Measured shapes and the implementation boundary

| Question | Actual repository location | Input → output |
|---|---|---|
| Where is raw data configured? | `configs/intel_welding/foundations.json` | repository-relative data root |
| Where are modality files found? | `scripts/intel_welding/04_foundations.py:checked_path`, `inspect_sample` | manifest SUBDIRS → FLAC/AVI/CSV paths |
| How are timestamps checked? | `scripts/intel_welding/04_foundations.py:timing` | Date+Time → cadence and validity |
| Where is the overview saved? | `artifacts/intel_welding/audits/foundation_runs/20260927T114705163655Z/` | inventory, alignment and label/session reports |
| Where are labels defined? | `docs/intel_welding/research_contract.md` | intended category; proposed Good vs non-Good task |
| Where is the trained multimodal input loader? | Not implemented in the inspected foundation pipeline | `src/intel_welding/` subpackages are placeholders; no model-input shape is invented |

This pipeline currently establishes structure and feasibility. Four preview frames are not a training batch. A future **batch** would group several model samples after a reviewed preprocessing design. Post-weld images remain offline-only, and this notebook does not claim a clean final holdout or model capability.

In [ ]:
for relative in ['docs/intel_welding/research_contract.md','src/intel_welding/README.md']:
    track(ROOT/relative)
track(AUDIT/'alignment_audit.md');track(AUDIT/'split_disposition.json')
(OUT/'config/selection.json').write_text(json.dumps({'config':cfg,'sample':sample_identity,'audio_seconds':AUDIO_SECONDS,'selection_rule':'lexicographically first complete readable development sample with valid timestamps and local files'},indent=2))
table(pd.DataFrame(shapes))
for relative,before in inputs.items():
    after=identity(ROOT/relative,ROOT)
    assert after['bytes']==before['bytes'] and after['mtime_ns']==before['mtime_ns'], relative
(OUT/'tables/shapes.json').write_text(json.dumps(shapes,indent=2))
(OUT/'manifests/inputs.json').write_text(json.dumps(list(inputs.values()),indent=2))
(OUT/'reports/verification.json').write_text(json.dumps({'inputs_metadata_unchanged':True,'sample':sample_identity,'warnings':warnings,'training_or_evaluation_run':False,'browser_playback_verified':False},indent=2))
display(Markdown('**Read-only checks passed.** Input size/mtime unchanged. No models trained or evaluated; no inventories, labels or feature banks regenerated.'))
for warning in warnings: display(Markdown('> '+warning))